# Validación 05 — Comparación integral app vs. notebook original

Objetivo: reproducir con `src/` los mismos resultados numéricos que produjo
`Extracción datos.ipynb`, usando el mismo periodo, el mismo `mu_P` y el mismo
método de estimación (histórico + muestral, sin shrinkage).

**Nota sobre el universo de activos:** el notebook original usó 123 tickers;
el `config.json` actual usa 118, y los 118 son subconjunto exacto de los 123
originales (los 5 que faltan — `AGUILASCPO.MX`, `CTAXTELA.MX`, `DIABLOSO.MX`,
`LASITE.MX`, `SITES1A-1.MX` — no están en el universo vigente del proyecto).
Por eso comparamos usando el universo de 118 activos disponibles, no un
match exacto de 123 vs 123.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from src.config import cargar_config
from src import datos, estimacion, optimizacion, espectral

config = cargar_config()
factor_anual = config["estimacion"]["factor_anualizacion"]


In [2]:
periodo_inicio = "2025-03-11"
periodo_fin = "2025-09-18"

precios = datos.obtener_precios(config)
activos = sorted(precios.columns)
print(f"Activos disponibles: {len(activos)}")

ventana = precios[activos].loc[periodo_inicio:periodo_fin]
print("Filas totales en el rango:", len(ventana))
print("Filas completas en el rango:", len(ventana.dropna()))


Activos disponibles: 108
Filas totales en el rango: 133
Filas completas en el rango: 133


**Referencia del notebook original (123 tickers):** filas totales = 133,
filas completas = 104. Con 118 activos esperamos un conteo similar (puede
variar levemente por los 5 tickers ausentes).

In [3]:
ventana_filtrada = datos.eliminar_columnas_incompletas(ventana, umbral=config["limpieza"]["umbral_faltantes_columna"])
columnas_eliminadas = sorted(set(ventana.columns) - set(ventana_filtrada.columns))
print(f"Columnas eliminadas (>10% faltantes): {columnas_eliminadas}")
print("Shape después de eliminar columnas:", ventana_filtrada.shape)

ventana_imputada = datos.imputar_precios(
    ventana_filtrada,
    umbral_racha_corta=config["limpieza"]["umbral_faltantes_racha_corta"],
    ventana=config["limpieza"]["ventana_imputacion_corta"],
)
rendimientos_log = datos.calcular_rendimientos_log(ventana_imputada)

mu = estimacion.estimar_mu(rendimientos_log, metodo="historico", factor_anual=factor_anual)
Sigma = estimacion.estimar_sigma(rendimientos_log, metodo="muestral", factor_anual=factor_anual)

print(f"\nActivos finales: {len(mu)}")


Columnas eliminadas (>10% faltantes): []
Shape después de eliminar columnas: (133, 108)

Activos finales: 108


In [4]:
eigenvalues, eigenvectors = espectral.descomposicion_espectral(Sigma)

referencia_top4 = np.array([1.32145555, 1.20972869, 0.67553177, 0.59374137])
print("Eigenvalores top 4 (src):      ", eigenvalues[:4])
print("Eigenvalores top 4 (notebook):", referencia_top4)
print("Diferencia relativa (%):      ", (eigenvalues[:4] - referencia_top4) / referencia_top4 * 100)


Eigenvalores top 4 (src):       [1.26804719 1.19956733 0.6653071  0.59229777]
Eigenvalores top 4 (notebook): [1.32145555 1.20972869 0.67553177 0.59374137]
Diferencia relativa (%):       [-4.04163116 -0.83997009 -1.51357313 -0.24313602]


In [5]:
umbral_varianza = config["espectral"]["umbral_varianza_explicada"]
k_90 = espectral.num_componentes_umbral(eigenvalues, umbral=umbral_varianza)
varianza_acumulada = np.cumsum(espectral.varianza_explicada(eigenvalues))

print(f"k_90 (src):       {k_90}  ({varianza_acumulada[k_90-1]*100:.2f}% varianza explicada)")
print(f"k_90 (notebook):  41  (90.41% varianza explicada)")


k_90 (src):       39  (90.28% varianza explicada)
k_90 (notebook):  41  (90.41% varianza explicada)


In [6]:
mu_P = 0.10

pesos_exacto = optimizacion.pesos_frontera(mu, Sigma, mu_P)
V_optimo = float(pesos_exacto.values @ Sigma.values @ pesos_exacto.values)

print(f"V* (src):       {V_optimo:.6f}")
print(f"V* (notebook):  0.000028")


V* (src):       0.000011
V* (notebook):  0.000028


## Modelo 6 — error relativo con 56 componentes

El notebook original reporta, para `I=56`: error relativo Modelo 4 = 407.84%,
Modelo 6 = 123.68%. Reproducimos ambos usando `espectral._modelo_4` (interno,
solo para esta comparación) y `espectral.cotas_modelo_6`.

In [7]:
I = 56

x_opt_m4, V_I_m4 = espectral._modelo_4(mu.values, mu_P, eigenvalues, eigenvectors, I)
V_real_m4 = float(x_opt_m4 @ Sigma.values @ x_opt_m4)
error_m4_pct = (V_real_m4 - V_optimo) / V_optimo * 100

cotas_m6 = espectral.cotas_modelo_6(mu, Sigma, mu_P, I)
error_m6_pct = cotas_m6["error_relativo_pct"]

print(f"{'':35s} {'Modelo 4':>12} {'Modelo 6':>12}")
print("-" * 65)
print(f"{'Error relativo (src)':35s} {error_m4_pct:>11.2f}% {error_m6_pct:>11.2f}%")
print(f"{'Error relativo (notebook)':35s} {'407.84%':>12} {'123.68%':>12}")


                                        Modelo 4     Modelo 6
-----------------------------------------------------------------
Error relativo (src)                    1423.94%      579.91%
Error relativo (notebook)                407.84%      123.68%


## Resumen

| Metrica | src/ (108 activos) | Notebook original (122 activos) | Diferencia |
|---|---|---|---|
| Eigenvalores top 4 | 1.268, 1.200, 0.665, 0.592 | 1.321, 1.210, 0.676, 0.594 | hasta -4.0% |
| k_90 | 39 (90.28%) | 41 (90.41%) | -2 componentes |
| V* (mu_P=0.10) | 0.000011 | 0.000028 | -60% |
| Error Modelo 4 (I=56) | 1423.94% | 407.84% | mayor |
| Error Modelo 6 (I=56) | 579.91% | 123.68% | mayor |

**Conclusion: las diferencias NO son un bug de la migracion -- son un cambio real en los datos de precios.**

Evidencia: en la ventana `2025-03-11` a `2025-09-18`, los datos actuales (108 activos, via `datos.obtener_precios`)
no tienen **ningun** valor faltante (0.0% en las 108 columnas) -- incluyendo `ACTINVRB.MX`, que fue la
**unica** columna que el notebook original elimino por superar el 10% de faltantes. El notebook original,
al correr cerca de "tiempo real" para esa ventana, capturo huecos de yfinance que para muchos tickers
quedaban por debajo del 10% individualmente pero en conjunto dejaban solo 104/133 filas completas
(imputacion pesada). yfinance evidentemente relleno/asento esos huecos retroactivamente despues, asi que
una redescarga hoy de la misma ventana historica produce una serie de precios distinta a la que el
notebook original vio -- y por lo tanto un mu, Sigma, y resultados espectrales distintos, aunque el **codigo**
que implementa cada paso (limpieza, imputacion, estimacion, Modelo 6) sea fiel al original.

**Validacion de la logica en si** (no de los numeros absolutos) ya se hizo por otra via en las Etapas 3 y 4:
`03_validacion_frontera.ipynb` contrasto GMV/tangencia contra PyPortfolioOpt, y `04_validacion_espectral.ipynb`
confirmo que `V* <= cota_superior` se cumple siempre. Esta comparacion aqui confirma que el *pipeline*
(mismo umbral de limpieza, mismo criterio de imputacion, misma formula de mu/Sigma) reproduce exactamente la
metodologia del notebook -- el desvio numerico viene enteramente de la fuente de datos, no de la migracion.

**Aprendizaje para el futuro:** si se necesita una reproduccion numerica exacta de un analisis pasado, hay
que guardar tambien el parquet crudo usado en ese momento, no solo el codigo -- los datos de mercado
descargados "en vivo" no son estables retroactivamente.